In [2]:
import json
import pandas as pd
import os
import duckdb
folder_path = "json_files/"

In [34]:
def empty_json():
    for item in os.listdir(folder_path):
        # Get the full path of the item
        full_path = os.path.join(folder_path, item)
        if full_path != 'json_files/key_info.json':
            empty = []
            with open(full_path, 'w') as file:
                json.dump(empty, file, indent=4)
            print('overwritten ', full_path)
        else:
            reset_key = [{
                            "last_pk": 0
                        }]
            with open(full_path, 'w') as file:
                json.dump(reset_key, file, indent=4)
            print('key reset')
            


In [53]:
empty_json()

overwritten  json_files/ISF.json
key reset
overwritten  json_files/location.json
overwritten  json_files/media.json
overwritten  json_files/personnel.json
overwritten  json_files/raw.json
overwritten  json_files/site_activity.json
overwritten  json_files/subsystem.json
overwritten  json_files/systembaseline.json
overwritten  json_files/test.json


In [54]:
with open(folder_path + 'systembaseline.json', 'r') as file:
    systembaseline_list = json.load(file)

In [55]:
with open(folder_path + 'site_activity.json', 'r') as file:
    site_activity_list = json.load(file)

In [56]:
systembaseline_list

[{'FK': 1, 'systembaseline': '4.7.3.2'}]

In [57]:
site_activity_list

[{'Timestamp': '2026-10-14T00:45:16.624000+08:00',
  'Date of Site Activity': '2026-10-14T00:00:00+08:00',
  'Start Time': '08:00',
  'End Time': '18:00',
  'Title': 'Title',
  'Activity Description': 'Testing',
  'Send': 1,
  'PK': 1}]

In [58]:
df_site_activity = pd.DataFrame(site_activity_list)

In [59]:
df_site_activity

,Timestamp,Date of Site Activity,Start Time,End Time,Title,Activity Description,Send,PK
0,2026-10-14T00:45:16.624000+08:00,2026-10-14T00:00:00+08:00,08:00,18:00,Title,Testing,1,1


In [60]:
new_columns = [df_site_activity.columns[-1]] + list(df_site_activity.columns[:-1])
df_site_activity = df_site_activity[new_columns]

In [75]:
df_site_activity.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   PK                     1 non-null      int64
 1   Timestamp              1 non-null      str  
 2   Date of Site Activity  1 non-null      str  
 3   Start Time             1 non-null      str  
 4   End Time               1 non-null      str  
 5   Title                  1 non-null      str  
 6   Activity Description   1 non-null      str  
 7   Send                   1 non-null      int64
dtypes: int64(2), str(6)
memory usage: 196.0 bytes


In [93]:
all_df = {}

for item in os.listdir(folder_path):
        # Get the full path of the item
    full_path = os.path.join(folder_path, item)
    print(full_path)
    with open(full_path, 'r') as file:
        new_table = json.load(file)
    idx = item.index('.')
    all_df[item[:idx]] = pd.DataFrame(new_table)
    all_df[item[:idx]].columns = all_df[item[:idx]].columns.str.replace(" ", "_")
    new_columns = [all_df[item[:idx]].columns[-1]] + list(all_df[item[:idx]].columns[:-1])
    all_df[item[:idx]] = all_df[item[:idx]][new_columns]

all_df.keys()

json_files/ISF.json
json_files/key_info.json
json_files/location.json
json_files/media.json
json_files/personnel.json
json_files/raw.json
json_files/site_activity.json
json_files/subsystem.json
json_files/systembaseline.json
json_files/test.json


dict_keys(['ISF', 'key_info', 'location', 'media', 'personnel', 'raw', 'site_activity', 'subsystem', 'systembaseline', 'test'])

In [94]:
all_df['raw']['Timestamp'] = pd.to_datetime(all_df['raw']['Timestamp'])
all_df['raw']['Date_of_Site_Activity'] = pd.to_datetime(all_df['raw']['Date_of_Site_Activity'])
all_df['site_activity']['Timestamp'] = pd.to_datetime(all_df['site_activity']['Timestamp'])
all_df['site_activity']['Date_of_Site_Activity'] = pd.to_datetime(all_df['site_activity']['Date_of_Site_Activity'])

In [95]:
locals().update(all_df)

In [96]:
query = """
SELECT team AS Team, name AS Name
FROM personnel
ORDER BY Team
"""
result_df = duckdb.query(query).to_df()
result_df

,Team,Name
0,J152,Erwin
1,J152,Pablo
2,JRLP,Jia Ling
3,JRLP,Herman
4,JRLP,Ern-Min


Search by Date

In [102]:
primary_key = 1
from_date = '2026-09-10'
to_date = '2026-12-31'
query = """
SELECT *
FROM site_activity
WHERE 1=1
AND PK = $primary_key
AND Date_of_Site_Activity > $from_date
AND Date_of_Site_Activity < $to_date

"""
result_df = duckdb.query(query, params={"primary_key": primary_key, "from_date": from_date, "to_date": to_date}).to_df()
result_df

,PK,Timestamp,Date_of_Site_Activity,Start_Time,End_Time,Title,Activity_Description,Send
0,1,2026-10-14 00:45:16.624000+08:00,2026-10-14 00:00:00+08:00,08:00,18:00,Title,Testing,1


In [101]:
query = """
SELECT * FROM site_activity
WHERE Date_of_Site_Activity = '2026-10-14'
"""
result_df = duckdb.query(query).to_df()
result_df

,PK,Timestamp,Date_of_Site_Activity,Start_Time,End_Time,Title,Activity_Description,Send
0,1,2026-10-14 00:45:16.624000+08:00,2026-10-14 00:00:00+08:00,08:00,18:00,Title,Testing,1
